# Distribution-guided quantum machine unlearning under data poisoning

This notebook runs and analyzes our Phase 1 study. We evaluate Malik et al.'s
existing class-level unlearning method on MNIST digits 0, 1, and 2. Digit 2 is
the forget class; digits 0 and 1 are retained.

Reusable code is in `src/`. All figures appear here and can be exported with
their tables. Completed results are loaded from their saved study indexes.
The default mode reviews saved results without training or generating predictions.
Missing results stop review; they are never filled in automatically.


## 1. Research question and success criteria

Does controlled poisoning change the forgetting, retained utility, and
retraining agreement of the existing distribution-guided method?

We compare the clean gold reference, poisoned originals, guided and uniform
unlearning, and matched retained-data retraining. We do not require the quantum
classifier to beat classical baselines. Lower digit-2 recall alone is insufficient:
also examine retained accuracy, forgotten-class confidence, and retraining KL.
An unfavorable or inconclusive result is valid evidence.


### 1.1 Reading the evidence

All evaluation images and labels stay clean. Compare poisoning damage before
unlearning with the additional change after unlearning within the same fold.
Report digits 0 and 1 separately. Five overlapping folds use one seed; their
standard deviation is descriptive variability, not uncertainty over five seeds.

The method follows Malik et al., *Distribution-Guided and Constrained Quantum
Machine Unlearning*. The real-vector corruption is inspired by Chen and Zhang,
*Superior resilience to poisoning and amenability to unlearning in quantum
machine learning*. Our four-PCA-feature, six-qubit task is an explicit adaptation.


## 2. Environment and GPU availability

### 2.1 Select the project and Python environment

Open this notebook from the project directory and select the environment with
`requirements.txt` installed. Restart the kernel before changing execution mode.
The notebook does not install or replace packages automatically.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src/config.py").is_file():
    raise RuntimeError("Open main_project.ipynb from the project root containing src/.")
if "src" in sys.modules:
    imported_root = Path(sys.modules["src"].__file__).resolve().parent.parent
    if imported_root != PROJECT_ROOT:
        raise RuntimeError("A different project is imported. Restart the kernel.")
sys.path.insert(0, str(PROJECT_ROOT))
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


### 2.2 Check the physical GPU and Aer support

A visible NVIDIA GPU does not guarantee that the installed Aer build supports
it. Windows CPU-only Aer can review all saved results and run local simulation.
GPU memory use is not a speed target for a six-qubit circuit.


In [ ]:
from src.artifacts import environment_info

environment = environment_info()
gpu_available_to_aer = "GPU" in environment["aer_devices"]
print("Operating system:", environment["platform"])
print("NVIDIA report:", environment["nvidia_gpu"] or "Not available")
print("Aer devices:", environment["aer_devices"])
print("Aer GPU support:", gpu_available_to_aer)


### 2.3 Import notebook helpers


In [ ]:
from dataclasses import asdict, replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import artifacts, project_workflow as workflow
from src.config import DEFAULT_SEED
from src.fresh_study import base_config, condition_configs, study_plan, PRIMARY_RATES, RATES
from src.evaluation import fresh_analysis
from src.evaluation.plots import (
    plot_examples, plot_class_balance, plot_pca_variance, plot_training_history,
    plot_confusion_comparison, plot_forget_confidence, draw_circuit,
)

display(pd.Series(environment["packages"], name="Installed version").to_frame())


## 3. Settings and execution controls

### 3.1 Fixed scientific settings

Use 420 MNIST training images, 140 per digit. Each of five outer folds has
268 training, 68 validation, and 84 test images. PCA and angle scaling are fitted
only on the relevant model's training records. Clean and corrupted vectors are
normalized to unit length before four-component PCA.

The primary study uses exact noiseless statevector expectations, seed 6304,
300 supervised updates, batch size 10, and 100 unlearning updates.


In [ ]:
SEED = DEFAULT_SEED  # 6304 for all runs and deterministic derived streams.
STUDY_FOLDS = (0, 1, 2, 3, 4)
ILLUSTRATION_FOLD = 0
STUDY_LABEL = "feature_randomization_v1"
config = base_config()

display(pd.Series({
    "seed": SEED, "images": 3 * config.data.train_per_class,
    "folds": len(STUDY_FOLDS), "PCA features": 4, "qubits": 6,
    "supervised updates": config.training.steps,
    "unlearning updates": config.unlearning.training.steps,
    "batch size": config.training.batch_size, "shots": config.backend.shots,
}, name="Protocol").to_frame())


### 3.2 Saved review or new local execution

Keep `SAVED_RESULTS_ONLY=True` to inspect imported completed results. Every fit,
download, force flag, and hardware flag must remain False in that mode. If the
study is missing, review stops with an explanation.

To create a new local experiment, restart the kernel, set
`SAVED_RESULTS_ONLY=False`, and enable the required stages. A new clone with no
data also needs `DOWNLOAD_MNIST=True`. Set `RUN_PRIMARY_STUDY=True` for the full
primary study. Completed compatible fits are reused; unfinished fits resume only
when their strict identity matches. Force flags deliberately create new attempts.


In [ ]:
SAVED_RESULTS_ONLY = True
DOWNLOAD_MNIST = False
RUN_PRIMARY_STUDY = False
RUN_ABLATIONS = False
RUN_FIXED_WEIGHT_NOISE = False
RUN_NOISY_TRAINING = False
FORCE_RETRAIN = False
FORCE_ABLATIONS = False
FORCE_NOISE_EVALUATION = False
FORCE_NOISY_TRAINING = False
USE_IBM_HARDWARE = False

STUDY_INDEX = 'results/fresh_studies/fa1604cdd38207d9/study.json'  # Verified completed study.
ABLATION_INDEX = 'results/ablations/e4a96e40f818e487/study.json'  # Verified completed study.
NOISE_INDEX = 'results/noise_studies/11aa94ebc456b91d/study.json'  # Verified completed study.
NOISY_TRAINING_INDEX = None


### 3.3 Execution and display preferences

Independent folds and fits can run concurrently through one Aer service. Each
optimizer keeps its own parameters, random stream, and checkpoint. CPU fallback
is selected from actual Aer support. These settings describe a new execution;
they never rewrite the recorded device of a saved experiment.


In [ ]:
DEVICE = "auto"
PARALLEL_WORKERS = None  # Conservative default: up to 2 on CPU, 8 on a supported GPU.
RUN_BASELINE_COMPATIBILITY = True
SHOW_ABLATION_ANALYSIS = True
SHOW_NOISE_ANALYSIS = True
SHOW_NOISY_TRAINING_ANALYSIS = False
SHOW_FULL_CHECKPOINT_HEALTH = False  # Optional recursive inventory; not needed to load models.
EXPORT_ANALYSIS = True

execution = None
study = None
primary_study = None
figures = {}
extension_tables = {}
extension_figures = {}


### 3.4 Verify the selected saved study

Saved review checks hashes, membership, settings, and parent/reference links.
It reads numeric artifacts without unpickling trained estimators. Training
provenance is preserved separately from this machine's review environment.
The review guard remains active until the kernel is restarted.


In [ ]:
execution_flags = dict(
    DOWNLOAD_MNIST=DOWNLOAD_MNIST, RUN_PRIMARY_STUDY=RUN_PRIMARY_STUDY,
    RUN_ABLATIONS=RUN_ABLATIONS, RUN_FIXED_WEIGHT_NOISE=RUN_FIXED_WEIGHT_NOISE,
    RUN_NOISY_TRAINING=RUN_NOISY_TRAINING, FORCE_RETRAIN=FORCE_RETRAIN,
    FORCE_ABLATIONS=FORCE_ABLATIONS, FORCE_NOISE_EVALUATION=FORCE_NOISE_EVALUATION,
    FORCE_NOISY_TRAINING=FORCE_NOISY_TRAINING, USE_IBM_HARDWARE=USE_IBM_HARDWARE,
)
if SAVED_RESULTS_ONLY:
    review_guard = workflow.protect_saved_review(**execution_flags)
if SAVED_RESULTS_ONLY or not RUN_PRIMARY_STUDY:
    study = workflow.load_completed_study(
        PROJECT_ROOT, STUDY_INDEX, label=STUDY_LABEL,
        progress=lambda message: print(message, flush=True),
    )
    config = workflow.study_config(study, ILLUSTRATION_FOLD)
    if tuple(sorted(study.split_paths)) != STUDY_FOLDS:
        raise ValueError("Saved fold coverage differs from STUDY_FOLDS.")


### 3.5 Check storage only when execution is enabled

Each complete optimizer update saves parameters, Adam moments, and RNG state.
Completed model artifacts are separately checksummed. A full inventory can be
slow on a large filesystem, so normal loading verifies only the required models.


In [ ]:
from src.checkpoint_storage import verify_checkpoint_storage, checkpoint_status

if any((RUN_PRIMARY_STUDY, RUN_ABLATIONS, RUN_FIXED_WEIGHT_NOISE, RUN_NOISY_TRAINING)):
    storage = verify_checkpoint_storage(PROJECT_ROOT)
    execution = workflow.local_execution(device=DEVICE, workers=PARALLEL_WORKERS)
    print("Storage verified:", storage["verified"])
    display(pd.Series(asdict(execution), name="New execution").to_frame())
if SHOW_FULL_CHECKPOINT_HEALTH:
    display(pd.DataFrame(checkpoint_status(PROJECT_ROOT)))


## 4. Data selection, folds, and preprocessing

### 4.1 Load raw images and exact memberships

The MNIST archive stays in `data/raw/`. Saved review restores the original
training/validation/test IDs and verifies their labels and image fingerprints.
It does not create a different split or refit PCA.


In [ ]:
if study is not None:
    splits = workflow.restore_study_splits(study)
else:
    splits = workflow.local_splits(
        PROJECT_ROOT, config, folds=STUDY_FOLDS, download=DOWNLOAD_MNIST,
    )
selected_split = splits[ILLUSTRATION_FOLD]


### 4.2 Check partition sizes and class balance


In [ ]:
split_counts = pd.DataFrame([
    {"fold": fold, "partition": name, "images": len(part),
     **{f"digit_{digit}": int((part.labels == digit).sum()) for digit in (0, 1, 2)}}
    for fold, split in splits.items()
    for name in ("train", "validation", "test")
    for part in (getattr(split, name),)
])
display(split_counts)


### 4.3 Interpret folds

Each image appears in one outer test fold. A model in a given fold sees only
that fold's training data during fitting. Validation selects its checkpoint;
test labels never select parameters or poisoning settings. Folds overlap in
training data, so they are not independent experimental seeds.


In [ ]:
figure = plot_class_balance({name: getattr(selected_split, name).labels
                             for name in ("train", "validation", "test")})
figures["selected_fold_class_balance"] = figure
display(figure)
plt.close(figure)


### 4.4 Inspect clean training examples


In [ ]:
figure = plot_examples(selected_split.train, max_examples=6)
figures["clean_training_examples"] = figure
display(figure)
plt.close(figure)


### 4.5 Four PCA features

The classifier receives four angles, not all 784 pixel values. Every original
model fits its own PCA and scaler on its training inputs. Retained-data
retraining fits a new transformer after excluding exact forget records.
Unlearning keeps its parent's numeric features frozen.


In [ ]:
from src.data.feature_randomization import UnitInputPreprocessor

if study is not None:
    features, pca_information, pca_view = workflow.cached_fold_features(study, ILLUSTRATION_FOLD)
else:
    preview_processor = UnitInputPreprocessor(SEED).fit(
        selected_split.train.images, selected_split.train.sample_ids,
    )
    features = {name: preview_processor.transform(getattr(selected_split, name).images)
                for name in ("train", "validation", "test")}
    pca_information = preview_processor.diagnostics()
    pca_view = preview_processor
display(pd.DataFrame([{ "partition": name, "shape": values.shape}
                      for name, values in features.items() if name in ("train", "validation", "test")]))


### 4.6 How much variance do four components preserve?


In [ ]:
display(pd.Series(pca_information, name="Saved or training-only PCA diagnostics").to_frame())
figure = plot_pca_variance(pca_view)
figures["pca_variance"] = figure
display(figure)
plt.close(figure)


**Interpretation:** high explained variance does not guarantee that the features
preserve all class information. PCA is fitted without test labels. A discarded
pixel change may have little effect on model inputs; inspect the paired
corruption diagnostic in Section 9 before interpreting attack effectiveness.


## 5. Classical baselines on the same inputs

### 5.1 Purpose and recorded models

Majority, logistic regression, linear SVM, and RBF SVM provide context for the
four-feature representation. They are not prerequisites that the gold VQC must
outperform. Their preprocessing and split match the clean original VQC in each
fold. Classical models are saved with supervised fits, so review does not refit them.


In [ ]:
gold_runs = workflow.clean_runs(study) if study is not None else {}
baseline_results = workflow.baseline_table(gold_runs)
if not baseline_results.empty:
    display(baseline_results.round(4))
else:
    print("Baselines will be available with the gold checkpoints in Section 7.")


### 5.2 Compare folds without pooling reused clean models


In [ ]:
if not baseline_results.empty:
    baseline_summary = baseline_results.groupby("model")[["accuracy", "retained_accuracy", "forget_recall"]].agg(
        ["mean", "std", "count"]
    )
    display(baseline_summary.round(4))


**Interpretation:** a gap between classical and quantum accuracy may reflect
the fixed architecture, optimizer, bounded logits, or representation. Fold
evaluation measures variation; it does not by itself explain the gap.


## 6. Quantum circuit and simulation budget

### 6.1 Draw the actual Malik-style circuit

Two feature-map reuploads each precede a three-repeat RY/RZ/CX ansatz. Six
qubits and 72 nominal angles are retained. Z expectations on qubits 3, 4, and 5
become three logits followed by softmax. This is an independent implementation
from the reported circuit and equations, not the authors' private repository.


In [ ]:
from src.models.circuits import build_malik_circuit, circuit_resources

circuit = build_malik_circuit(config.circuit)
figure = draw_circuit(circuit, fold=45)
figures["circuit"] = figure
display(figure)
plt.close(figure)


### 6.2 Count the resources


In [ ]:
display(pd.Series(circuit_resources(circuit), name="Logical circuit").to_frame())
parameter_count = config.circuit.n_parameters
display(pd.Series({
    "nominal parameters": parameter_count,
    "expectation circuits per sample and full gradient": 1 + 2 * parameter_count,
    "batch size": config.training.batch_size,
    "supervised updates per fit": config.training.steps,
    "unlearning updates per fit": config.unlearning.training.steps,
}, name="Optimization work").to_frame())


### 6.3 Check the saved prediction format

Review reads existing three-class probabilities and checks their normalization.
It does not simulate a forward pass again. Training uses expectation-level
parameter shifts and the classical softmax/cross-entropy chain rule.


In [ ]:
if gold_runs:
    saved_probabilities = gold_runs[ILLUSTRATION_FOLD]["predictions"]["model__test"]
    np.testing.assert_allclose(saved_probabilities.sum(axis=1), 1.0, atol=1e-10)
    display(pd.DataFrame(saved_probabilities[:5], columns=["p(0)", "p(1)", "p(2)"]))


## 7. Clean gold reference model

### 7.1 Train or load the clean originals

The gold reference is the unpoisoned three-class classifier. In new execution,
fit clean originals across selected folds first. The main pipeline reuses these
same cache identities. Forced experiments defer this step to the full pipeline
so the gold models are fitted only once in the new attempt.


In [ ]:
def show_progress(event):
    if "completed" in event:
        print(f"Completed {event['completed']}/{event['planned']}: {event.get('problem', '')}", flush=True)
    elif event.get("step") is not None:
        print(f"fold {event['fold']} {event['condition']} {event['role']}: "
              f"{event['step']}/{event['updates']}", flush=True)

if RUN_PRIMARY_STUDY and not FORCE_RETRAIN:
    gold_runs = workflow.train_gold_models(
        PROJECT_ROOT, splits, execution, config=config, label=STUDY_LABEL, progress=show_progress,
    )
gold = gold_runs.get(ILLUSTRATION_FOLD)


### 7.2 Learning curve and validation-selected checkpoint


In [ ]:
if gold is not None:
    display(gold["history"].tail().round(5))
    figure = plot_training_history(gold["history"], title="Gold reference: supervised loss")
    figures["gold_training_history"] = figure
    display(figure)
    plt.close(figure)


### 7.3 Gold versus classical baselines


In [ ]:
baseline_results = workflow.baseline_table(gold_runs)
if not baseline_results.empty:
    display(baseline_results.round(4))
    display(baseline_results.groupby("model")[["accuracy", "retained_accuracy", "forget_recall"]].agg(
        ["mean", "std", "count"]
    ).round(4))


### 7.4 Digit-wise errors


In [ ]:
if gold is not None:
    gold_metrics = gold["metrics"]["models"]["model"]
    display(pd.Series(gold_metrics["recall_by_class"], name="Test recall by digit").to_frame())
    display(pd.Series({name: gold_metrics[name] for name in
                       ("accuracy", "retained_accuracy", "forget_recall", "forget_confidence")},
                      name="Gold reference").to_frame())


### 7.5 Confusion matrix


In [ ]:
if gold is not None:
    from sklearn.metrics import ConfusionMatrixDisplay
    figure, axis = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(np.asarray(gold_metrics["confusion_matrix"]), display_labels=[0, 1, 2]).plot(
        ax=axis, colorbar=False,
    )
    axis.set_title(f"Gold reference: fold {ILLUSTRATION_FOLD}")
    figures["gold_confusion"] = figure
    display(figure)
    plt.close(figure)


### 7.6 Recorded execution resources


In [ ]:
if gold is not None:
    gold_resources = artifacts.load_json(gold["path"] / "resources.json")
    display(pd.Series(gold_resources, name="Recorded training resources").to_frame())
    display(pd.Series(artifacts.load_json(gold["path"] / "checkpoints/model.json"),
                      name="Selected checkpoint").to_frame())


## 8. Baseline interpretation and output handling

The clean gold model establishes utility before poisoning and unlearning.
Its accuracy is not evidence of successful forgetting. No new research scores
are supplied by this notebook until saved artifacts are restored or a stage is
actually executed. Figures are kept in memory for the separate exports in Section 14.


In [ ]:
display(pd.DataFrame([
    {"item": "Gold originals available", "count": len(gold_runs)},
    {"item": "Planned folds", "count": len(STUDY_FOLDS)},
    {"item": "Inline figures prepared", "count": len(figures)},
]))


## 9. Poisoned-data preparation

### 9.1 Conditions and rate denominator

Use clean training, digit-2 feature randomization, and label flipping from 0 to 2.
Primary rates are 5%, 10%, 20%, 30%, 40%, and 50%. Additional 25% conditions
support the ablations. Rates refer to the eligible class, rounded down, so the
percentage of the entire training set is smaller. Validation and test stay clean.


In [ ]:
from src.data.feature_randomization import feature_variant, experiment_config, INPUT_POLICY
from src.data.poisoning import forget_mask

configs = condition_configs(config, STUDY_FOLDS)
if study is not None:
    configs = {(c["config"]["data"]["fold"], name.split("_", 2)[2]): experiment_config(c["config"])
               for name, c in study.conditions.items()}
variants = {
    name: (workflow.restore_poison_variant(study, f"fold_{fold}_{name}", split=selected_split)
           if study is not None else feature_variant(selected_split.train, settings.poison))
    for (fold, name), settings in configs.items() if fold == ILLUSTRATION_FOLD
}
display(pd.DataFrame([{"condition": name, **variant.metadata()} for name, variant in variants.items()])[
    ["condition", "n_training", "eligible_count", "poisoned_count", "actual_fraction_eligible", "actual_fraction_all_training"]
])


### 9.2 Preserve original identities

Every record keeps its sample ID, original label, observed label, poison flag,
and seed. A selected record is corrupted only in training. The saved annotated
dataset can therefore be matched to the exact fold and poisoning setting.


In [ ]:
variant = variants["feature_randomization_50pct"]
display(variant.manifest().head(10))
assert set(variant.observed.sample_ids) == set(selected_split.train.sample_ids)
assert not set(variant.observed.sample_ids) & set(selected_split.test.sample_ids)


### 9.3 Feature randomization

For selected digit-2 records, replace the complete 784-dimensional input with
a real Gaussian vector, then normalize it to unit length. Labels remain 2.
Signed entries are intentional; this is a feature surrogate, not a natural
grayscale image. Do not clip its negative values before PCA. All inputs use
the same unit-normalization policy. The previous corner patch is superseded.


In [ ]:
display(pd.Series(INPUT_POLICY, name="Input policy").to_frame())
figure = fresh_analysis.plot_feature_examples(variants["feature_randomization_50pct"])
figures["feature_randomization_examples"] = figure
display(figure)
plt.close(figure)


### 9.4 Label flipping

Selected digit-0 images keep their pixels and IDs, but their observed label
becomes 2. Pixel differences are therefore zero for this attack. It changes
the feature-label association rather than the input features.


In [ ]:
figure = fresh_analysis.plot_feature_examples(variants["label_flip_50pct"])
figures["label_flip_examples"] = figure
display(figure)
plt.close(figure)


### 9.5 Verify unchanged validation and test data


In [ ]:
for name, variant in variants.items():
    np.testing.assert_array_equal(variant.original.sample_ids, variant.observed.sample_ids)
    if variant.config.kind == "label_flip":
        np.testing.assert_array_equal(variant.original.images, variant.observed.images)
    else:
        np.testing.assert_array_equal(variant.original.labels, variant.observed.labels)
print("Training-only variants aligned. Validation and test objects were not modified.")


### 9.6 Inspect observed class balance


In [ ]:
poison_balance = pd.DataFrame([
    {"condition": name, **{f"observed_digit_{digit}": int((variant.observed.labels == digit).sum())
                           for digit in (0, 1, 2)}}
    for name, variant in variants.items()
])
display(poison_balance)


### 9.7 Does the corruption reach four-angle model inputs?

This paired diagnostic transforms clean and corrupted records through frozen
clean PCA. Label flipping must have zero input change. Feature randomization
should change inputs, but a nonzero change does not establish harmful training.
Saved review loads the recorded diagnostic, and never refits a missing one.


In [ ]:
if study is not None:
    input_effects = workflow.saved_input_diagnostics(study)
else:
    input_effects, _ = fresh_analysis.input_diagnostics(configs, splits)
if input_effects.empty:
    print("No saved input diagnostic is available. Review will not refit PCA to replace it.")
else:
    display(input_effects.round(5))


### 9.7.1 Plot the paired feature changes


In [ ]:
if not input_effects.empty:
    figure = fresh_analysis.diagnostic_figure(input_effects)
    figures["input_changes"] = figure
    display(figure)
    plt.close(figure)


### 9.8 Exact forget-set membership

Forgetting is class-level: all observed-label-2 training records are designated
for forgetting. Under label flipping, this includes the flipped zeros.
Matched retained-data retraining removes these exact records. This experiment
does not identify and erase only the poisoned samples.


In [ ]:
forget_membership = pd.DataFrame([
    {"condition": name, "forget_records": int(forget_mask(variant).sum()),
     "retained_records": int((~forget_mask(variant)).sum()),
     "poisoned_forget_records": int((forget_mask(variant) & variant.is_poisoned).sum())}
    for name, variant in variants.items()
])
display(forget_membership)


### 9.9 Storage locations

The training runner creates `data/annotated/`, `data/splits/`, model run folders,
and condition manifests on demand. Review reconstructs previews in memory
and does not overwrite saved training data. Raw MNIST remains in `data/raw/`.


In [ ]:
if study is not None:
    condition = study.conditions[f"fold_{ILLUSTRATION_FOLD}_label_flip_50pct"]
    display(pd.Series({name: condition[name] for name in ("split_manifest", "annotated_dataset")},
                      name="Saved data links").to_frame())


### 9.10 Interpretation before model evaluation

Attack strength must be measured from the trained models, not inferred from
the appearance of corrupted inputs. A robust original classifier is a valid
finding. Keep all tested conditions rather than selecting only damaging ones.


## 10. Poisoned training and matched retained-data references

### 10.1 Planned fitting work

The primary pipeline also prepares 25% support conditions for ablations.
Identical clean retained-data problems are cached once per fold. All fits
start from reproducible initialization; guided and uniform methods later
restart from the same trained original.


In [ ]:
primary_plan = study_plan(config, STUDY_FOLDS)
display(primary_plan.groupby(["stage", "updates"]).size().rename("planned_fits").to_frame())
print("Planned primary and support fits:", len(primary_plan))
print("Counts describe planned work, not completed experiments.")


### 10.2 Preprocessing of the retraining references

Each retained-data reference refits PCA and scaling on its own remaining
records. Feature randomization preserves labels, so its retained training
problem equals the clean retained problem and is reused. Label flips remove
some additional original zeros, so their retraining references can differ.
Unlearning uses the original model's features and fixed teachers throughout.


### 10.3 Complete or load the primary pipeline

This runner stages the remaining original/reference fits followed by default
guided and uniform unlearning (100 updates each). The complete sequence uses
one protocol and separate per-model checkpoints. Existing clean gold fits from
Section 7 are reused. With training disabled, this cell only selects the saved
study that was already verified in Section 3.

The guided objective is forget-target cross-entropy + alpha times soft-anchor
cross-entropy + lambda times squared distance to the original parameters.
Its global target and original anchor predictions stay fixed. Section 11
inspects those targets and the resulting tradeoffs.


In [ ]:
if RUN_PRIMARY_STUDY:
    from src.fresh_study import run_fresh_study
    study, study_index = run_fresh_study(
        PROJECT_ROOT, splits, execution, base=config, label=STUDY_LABEL,
        force_retrain=FORCE_RETRAIN, progress=show_progress,
    )
primary_study = study.select_rates(PRIMARY_RATES)
gold_runs = workflow.clean_runs(study)
gold = gold_runs[ILLUSTRATION_FOLD]
print("Completed unique cached fits:", len(study.runs))
print("Study:", study.report_path.relative_to(PROJECT_ROOT))


### 10.4 Build tables from saved numeric results


In [ ]:
tables = fresh_analysis.study_tables(primary_study)
selected_metrics = tables["metrics"][tables["metrics"].fold == ILLUSTRATION_FOLD]
original_roles = ("clean", "poisoned", "retrained_clean", "retrained_poisoned")
display(selected_metrics[selected_metrics.model.isin(original_roles)][
    ["attack", "poison_rate", "model", "accuracy", "retained_accuracy", "forget_recall", "forget_confidence"]
].round(4))


### 10.5 Separate poisoning damage from unlearning changes


In [ ]:
display(tables["effects"][tables["effects"].fold == ILLUSTRATION_FOLD][
    ["attack", "poison_rate", "method", "metric", "poisoning_damage_before",
     "clean_unlearning_change", "poisoned_unlearning_change", "additional_change_under_poisoning"]
].round(4))


**Interpretation:** a poisoned model can already have poor digit-2 recall
before unlearning. Attribute only the paired before-to-after change to the
unlearning step. A retained-data reference is an evaluation target, not the
unpoisoned three-class gold reference.


## 11. Guided and uniform unlearning

### 11.1 Fixed targets and retained anchors

For calibration records from the training forget set, average the original
probabilities, remove class 2, raise retained entries to beta, and normalize.
This gives one global target q. Cache the original soft predictions on a
training-only retained anchor subset. Defaults: alpha=1, lambda=0.01, beta=1,
anchor fraction=0.25. Uniform uses q=(0.5, 0.5, 0).


In [ ]:
display(tables["targets"][tables["targets"].fold == ILLUSTRATION_FOLD].round(5))


### 11.2 Checkpoints and teacher quality

Guided and uniform start from identical original weights. Teachers are frozen
before optimization, not refreshed from the changing student. The anchor term
preserves the original behavior, including any errors it already contains.


In [ ]:
display(tables["checkpoints"][tables["checkpoints"].fold == ILLUSTRATION_FOLD].head(16))
display(tables["targets"].groupby(["attack", "poison_rate", "model"])[
    ["target_0", "target_1", "target_2"]
].mean().round(5))


### 11.3 Retained utility, forgetting, and retraining agreement


In [ ]:
unlearning_roles = ("poisoned_guided", "poisoned_uniform")
display(selected_metrics[selected_metrics.model.isin(unlearning_roles)][
    ["attack", "poison_rate", "model", "retained_accuracy", "forget_recall", "forget_confidence"]
].round(4))
display(tables["retraining"][tables["retraining"].fold == ILLUSTRATION_FOLD].round(5))


### 11.4 Trajectories and confusion matrices

Inspect clean, 50% label-flip, and 50% feature-randomization conditions in the
illustration fold. The left loss axis is supervised validation loss; the right
axis is the unlearning objective. Their scales have different meanings.


In [ ]:
for name, figure in fresh_analysis.detailed_figures(primary_study, ILLUSTRATION_FOLD).items():
    figures[name] = figure
    display(figure)
    plt.close(figure)


### 11.5 Forgotten-class confidence on clean test examples


In [ ]:
condition = primary_study.conditions[f"fold_{ILLUSTRATION_FOLD}_label_flip_50pct"]
before = condition["models"]["poisoned"]["predictions"]
after = condition["unlearned"]["poisoned_guided"]["predictions"]
figure = plot_forget_confidence(before["test_labels"], before["model__test"], after["model__test"])
figures["label_flip_guided_forget_confidence"] = figure
display(figure)
plt.close(figure)


### 11.6 Interpret the tradeoff

Lower forget recall and confidence, preserved digits 0/1, and better agreement
with matched retraining jointly support useful class-level forgetting. Softmax
of three logits bounded in [-1,1] cannot give exactly zero class probability.
Low conditional retained-class KL can also hide excess class-2 mass, so report
full KL and forgotten mass on retained records where available.


## 12. All-fold evidence and parameter/noise studies

### 12.1 Aggregate the five folds

Show mean, descriptive fold SD, valid fold count, and planned coverage. Shared
clean references are not extra observations. The 25% support conditions remain
separate from the six-rate primary curves.


In [ ]:
fold_summary = tables["fold_summary"]
fold_effects = tables["fold_effects"]
quantum_roles = ("poisoned", "poisoned_guided", "poisoned_uniform", "retrained_poisoned")
display(fold_summary[fold_summary.model.isin(quantum_roles)].round(4))
display(fold_effects.round(4))
print("Incomplete metric rows:", int((~fold_summary.complete_folds).sum()))


### 12.1.1 Digit-wise recall and paired poisoning effects


In [ ]:
display(fold_summary[fold_summary.model.isin(quantum_roles) &
                     fold_summary.metric.str.startswith("recall_digit_")].round(4))
display(tables["effects"].groupby(["attack", "poison_rate", "method", "metric"])[
    "additional_change_under_poisoning"
].agg(["mean", "std", "count"]).round(4))


### 12.1.2 Plot the primary poisoning curves


In [ ]:
for name, figure in fresh_analysis.comparison_figures(tables).items():
    figures[f"primary_{name}"] = figure
    display(figure)
    plt.close(figure)


### 12.1.3 Locate saved extension studies


In [ ]:
extension_paths, extension_status = workflow.extension_inventory(study, {
    "ablations": ABLATION_INDEX, "noise_studies": NOISE_INDEX,
    "noisy_training": NOISY_TRAINING_INDEX,
})
display(extension_status)
ablation_index = extension_paths["ablations"]
noise_index = extension_paths["noise_studies"]
noisy_index = extension_paths["noisy_training"]


### 12.2 One-factor-at-a-time ablations

Vary guided unlearning only, keeping the parents, splits, PCA, seed, anchors,
and 100-update budget fixed. Conditions: clean, both attacks at 25%, and both
attacks at 50%, across all five folds. The default guided checkpoint and the
uniform comparison are reused rather than retrained for each parameter.

Alpha: 0, 0.5, 1, 2, 4. Lambda: 0, 0.001, 0.01, 0.1.
Beta: 0.25, 0.5, 0.75, 1, 2, 4. The extended values are our stress tests;
Malik's reported values are documented in the research context.


In [ ]:
from src.ablations import SWEEPS, DEFAULTS, plan_table, ablation_execution, run_study
from src.evaluation import ablation_analysis

ablation_problems = study.ablation_problems(STUDY_FOLDS)
ablation_plan = plan_table(ablation_problems)
display(ablation_plan.groupby(["parameter", "value"]).size().rename("planned_fits").to_frame())
print("Nondefault ablation fits:", len(ablation_plan))


### 12.2.1 Execute only when explicitly enabled

Alpha=0 removes retained-anchor loss; lambda=0 removes parameter preservation.
No Cartesian grid is used. Optional first-fold compatibility checks are kept
separate from the 300 nondefault parameter fits.


In [ ]:
if RUN_ABLATIONS:
    with ablation_execution(PROJECT_ROOT):
        ablation_index = run_study(
            study, ablation_problems, execution, force_retrain=FORCE_ABLATIONS,
            baseline_checks=RUN_BASELINE_COMPATIBILITY,
            progress=lambda event: print(f"Ablations {event['completed']}/{event['planned']}: "
                f"{event['problem']} {event['parameter']}={event['value']}", flush=True),
        )


### 12.2.2 Parameter summary, target distributions, and paired changes


In [ ]:
if SHOW_ABLATION_ANALYSIS and ablation_index is not None:
    print("Reading saved ablation metrics...", flush=True)
    _, ablation_results = ablation_analysis.study_tables(ablation_index, PROJECT_ROOT)
    extension_tables["ablations"] = ablation_results
    display(ablation_results["fold_summary"].round(4))
    display(ablation_results["paired_fold_summary"].round(4))
    display(ablation_results["summary"][["condition", "fold", "parameter", "value", "target_0", "target_1", "target_2"]].round(5))
    display(ablation_results["coverage"].groupby("completed").size().rename("fits").to_frame())
else:
    print("Saved ablation analysis is disabled or its completed index is not available.")


### 12.2.3 Plot each parameter's effect


In [ ]:
if "ablations" in extension_tables:
    extension_figures["ablations"] = ablation_analysis.comparison_figures(extension_tables["ablations"])
    for figure in extension_figures["ablations"].values():
        display(figure)
        plt.close(figure)


**Interpretation:** compare each setting with its own fold-matched default.
Report retention/forgetting tradeoffs, not only the smallest digit-2 recall.
Beta changes the target distribution; a nearly uniform teacher can make beta
changes small. Test-set plots are evaluation, not a tuning rule.


### 12.3 Scope of additional controls

Shuffled-label fits, multiple training seeds, new circuits, and a new unlearning
method are outside this run. Classical baselines, the uniform target, alpha=0,
lambda=0, matched retraining, and zero-noise checks provide the current controls.


### 12.4 Fixed-weight synthetic noise evaluation

Keep trained weights fixed and evaluate all primary poisoning conditions
across five folds. Use zero noise and depolarizing, bit-flip, phase-flip,
readout, and combined profiles at p=0.001, 0.005, 0.01, 0.02. Gate strength is
p on one-qubit gates and 2p on CX; combined adds symmetric readout flips at p.
These are declared synthetic channels, not calibrated hardware noise.


In [ ]:
from src.noise_study import load_noise_inputs, noise_plan, run_noise_study
from src.models.noise import NoiseExecution
from src.evaluation import noise_analysis

if RUN_FIXED_WEIGHT_NOISE:
    noise_inputs, noise_bindings = load_noise_inputs(primary_study, STUDY_FOLDS, "all")
    display(noise_plan(noise_inputs, noise_bindings))
    noise_execution = NoiseExecution(device=execution.device,
        circuit_batch_size=512, threads=execution.threads, memory_mb=execution.memory_mb)


### 12.4.1 Generate new noise predictions only when enabled


In [ ]:
if RUN_FIXED_WEIGHT_NOISE:
    noise_index = run_noise_study(
        primary_study, noise_inputs, noise_bindings, noise_execution,
        allow_execution=True, force_evaluate=FORCE_NOISE_EVALUATION,
        progress=lambda event: print(f"Noise evaluation {event['completed']}/{event['planned']}", flush=True),
    )


### 12.4.2 Read saved noise metrics

This can read thousands of prediction files. It does not train models. The
export step will reuse these loaded tables instead of reading them again.
Noise comparisons use the retraining reference under the matched noise profile.


In [ ]:
if SHOW_NOISE_ANALYSIS and noise_index is not None:
    print("Reading and checking saved noise predictions; no model execution...", flush=True)
    _, noise_results = noise_analysis.noise_tables(noise_index, PROJECT_ROOT)
    extension_tables["noise_studies"] = noise_results
    display(noise_results["fold_summary"].round(4))
    display(noise_results["coverage"].groupby("completed").size().rename("rows").to_frame())
else:
    print("Saved fixed-weight noise analysis is disabled or not available.")


### 12.4.3 Compare retention and forgetting under noise


In [ ]:
if "noise_studies" in extension_tables:
    extension_figures["noise_studies"] = noise_analysis.noise_figures(extension_tables["noise_studies"])
    for figure in extension_figures["noise_studies"].values():
        display(figure)
        plt.close(figure)


### 12.5 Exact density probabilities and finite readout shots

Gate noise is integrated with density-matrix evolution. Exact readout
probabilities and a separate 2048-shot draw are both evaluated. Shot draws
are not new training seeds. Noise can reduce digit-2 confidence without
performing useful unlearning; always inspect retained utility and matched KL.


In [ ]:
if "noise_studies" in extension_tables:
    noise_summary = extension_tables["noise_studies"]["summary"]
    display(noise_summary.head(12).round(4))


### 12.6 Training and unlearning with noise inside optimization

Fit separate originals and retained-data references with combined gate/readout
noise at p=0.01. Freeze new noisy teachers, then run new guided and uniform
unlearning from the noisy originals. Cover all primary conditions and folds;
25% support conditions belong only to the ablations. This is 230 planned fits
for five folds (100 supervised and 130 unlearning).

Optimization uses exact noisy expectations, not sampled gradients. Compare
noiseless-trained and noisy-trained models under matched ideal/noisy deployment.


In [ ]:
from src.models.noise import NoiseSetting
from src.noisy_training import load_training_problems, training_plan, noisy_training_execution, run_noisy_training
from src.evaluation import noisy_training_analysis

training_noise = NoiseSetting("combined", 0.01)
display(pd.Series(training_noise.describe(), name="Training noise").to_frame())
display(study_plan(config, STUDY_FOLDS, PRIMARY_RATES).groupby(["stage", "updates"]).size().rename("planned_fits").to_frame())


### 12.6.1 Load inputs only if new noisy fitting is enabled

An unfinished optimizer cannot be resumed under changed source, environment,
or execution settings. Completed numeric results remain portable for review.
If an imported unfinished run is incompatible, the trainer stops rather than
quietly replacing it with a new experiment.


In [ ]:
if RUN_NOISY_TRAINING:
    print("Loading exact saved inputs for noisy training...", flush=True)
    noisy_problems, noisy_inputs, noisy_bindings = load_training_problems(primary_study, STUDY_FOLDS, "all")
    display(training_plan(noisy_problems).groupby(["stage", "updates"]).size().rename("fits").to_frame())


### 12.6.2 Continue compatible noisy fits with visible progress

The helper preserves an existing compatible execution before considering a
new benchmark. It reports model starts, optimizer steps, 30-second heartbeats,
and Aer activity. A completion count advances only after a full model is saved.
Slow updates and a stalled service are distinguished in its runtime logs.


In [ ]:
if RUN_NOISY_TRAINING:
    from tools.noisy_training_runtime import prepare_noisy_execution, noisy_training_monitor
    noisy_execution, noisy_benchmark, noisy_performance = prepare_noisy_execution(
        primary_study, noisy_problems, noisy_inputs, noisy_bindings, training_noise,
        selected_index=noisy_index, require_gpu_device=execution.device == "GPU",
    )
    display(noisy_benchmark)
    with noisy_training_execution(PROJECT_ROOT):
        with noisy_training_monitor(PROJECT_ROOT, heartbeat_seconds=30):
            noisy_index = run_noisy_training(
                primary_study, noisy_problems, noisy_inputs, noisy_bindings, noisy_execution,
                setting=training_noise, allow_training=True, force_retrain=FORCE_NOISY_TRAINING,
                performance_directory=noisy_performance,
                progress=lambda event: print(f"Noisy fits {event['completed']}/{event['planned']}", flush=True),
            )


### 12.6.3 Review completed noisy models and missing coverage

A partial study may contain valid completed models even when later fits are
unfinished. Report its coverage explicitly. Missing models or deployment
predictions are never replaced by invented scores or automatic fitting.


In [ ]:
if SHOW_NOISY_TRAINING_ANALYSIS and noisy_index is not None:
    print("Reading completed noisy fits and deployment predictions...", flush=True)
    _, noisy_results = noisy_training_analysis.noisy_training_tables(noisy_index, PROJECT_ROOT)
    extension_tables["noisy_training"] = noisy_results
    display(noisy_results["fold_summary"].round(4))
    display(noisy_results["coverage"])
else:
    print("Saved noisy-training analysis is disabled or not available.")


### 12.6.4 Plot matched deployment comparisons


In [ ]:
if "noisy_training" in extension_tables and not extension_tables["noisy_training"]["summary"].empty:
    extension_figures["noisy_training"] = noisy_training_analysis.noisy_training_figures(extension_tables["noisy_training"])
    for figure in extension_figures["noisy_training"].values():
        display(figure)
        plt.close(figure)


## 13. Optional IBM execution

Local Aer is sufficient for this protocol. IBM execution is disabled and no
account access is required. Hardware jobs need a separately declared backend,
finite shots, and explicit authorization; no hardware results are implied here.


In [ ]:
if USE_IBM_HARDWARE:
    raise RuntimeError("Declare a separate hardware protocol before enabling IBM execution.")
print("Primary protocol: local Aer simulation or saved-results review.")


## 14. Exports, provenance, and interpretation

### 14.1 Coverage and checkpoint roles

The gold reference is clean three-class training. Retained-data retraining
references remove the exact forget set. Original, guided, uniform, and
retraining roles link to immutable completed model artifacts. Missing extension
studies remain visibly missing and do not invalidate an independently complete
primary study.


In [ ]:
display(tables["resources"].round(3))
display(fold_summary.groupby("complete_folds").size().rename("metric_rows").to_frame())
study_paths = {"primary": study.report_path, "ablations": ablation_index,
               "fixed_weight_noise": noise_index, "noisy_training": noisy_index}
display(pd.DataFrame([{"study": name, "index": path.relative_to(PROJECT_ROOT).as_posix() if path else None}
                      for name, path in study_paths.items()]))


### 14.2 Preserve original training provenance

The machine displaying results may differ from the machine that trained them.
Recorded source hashes, device, execution, environment, and original paths stay
unchanged. The current review has its own source fingerprint and output folder.


In [ ]:
display(workflow.provenance_table(study, environment))
analysis_directory = workflow.review_directory(study)
print("Review exports:", analysis_directory.relative_to(PROJECT_ROOT))


### 14.3 Save tables and all displayed figures


In [ ]:
if EXPORT_ANALYSIS:
    fresh_analysis.export_study(primary_study, tables, figures, input_effects,
                                destination=analysis_directory / "primary")
    if "ablations" in extension_tables:
        ablation_analysis.export_study(ablation_index, PROJECT_ROOT,
            extension_figures.get("ablations"), tables=extension_tables["ablations"],
            destination=analysis_directory / "ablations")
    if "noise_studies" in extension_tables:
        noise_analysis.export_noise_study(noise_index, PROJECT_ROOT,
            extension_figures.get("noise_studies"), tables=extension_tables["noise_studies"],
            destination=analysis_directory / "fixed_weight_noise")
    if "noisy_training" in extension_tables:
        noisy_training_analysis.export_noisy_training(noisy_index, PROJECT_ROOT,
            extension_figures.get("noisy_training"), tables=extension_tables["noisy_training"],
            destination=analysis_directory / "noisy_training")
    print("Analysis exports saved.")


### 14.4 Link indexes and record this review


In [ ]:
if EXPORT_ANALYSIS:
    artifacts.save_json(analysis_directory / "review.json", {
        "studies": {name: path.relative_to(PROJECT_ROOT).as_posix() if path else None
                    for name, path in study_paths.items()},
        "original_training_protocol": study.index["protocol"],
        "current_source_fingerprint": artifacts.source_fingerprint(),
        "current_environment": environment,
        "saved_results_only": SAVED_RESULTS_ONLY,
        "planned_folds": STUDY_FOLDS, "seed": SEED,
    })


### 14.5 What can we conclude?

First establish the original poisoning effect. Then compare guided and uniform
unlearning using within-fold changes in retained accuracy, digit-2 recall and
confidence, and both retraining KL definitions. Check whether the pattern is
consistent across folds and parameter values. Noise robustness and noisy
optimization answer different questions and must be reported separately.

Record the observed direction, effect size, fold coverage, and limitations
after inspecting actual restored results. No conclusion is prefilled. This
single-seed study does not certify deletion or universal quantum superiority.


## 15. Phase 2: deferred

New targets, constraints, selective poisoned-record forgetting, and new
algorithms are outside the current evaluation. This cell is intentionally empty.
